# 06 Explain Plans

Read plans for scans, filters, joins, aggregations, exchanges, and adaptive execution.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("module-02-spark-sql").getOrCreate()
base = "../../datasets/module_02"

In [2]:
municipalities = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/municipalities.csv")
accessibility = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/accessibility_scores.csv")
poi = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/poi_counts.csv")
property_values = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/property_values.csv")
municipalities.printSchema()
municipalities.show(5, truncate=False)

root
 |-- municipality_id: integer (nullable = true)
 |-- municipality_name: string (nullable = true)
 |-- canton: string (nullable = true)
 |-- population: integer (nullable = true)

+---------------+-----------------+------+----------+
|municipality_id|municipality_name|canton|population|
+---------------+-----------------+------+----------+
|1              |Zurich           |ZH    |421878    |
|2              |Winterthur       |ZH    |116122    |
|3              |Uster            |ZH    |36012     |
|4              |Meilen           |ZH    |14733     |
|5              |Bern             |BE    |134591    |
+---------------+-----------------+------+----------+
only showing top 5 rows



In [3]:
municipalities.createOrReplaceTempView("municipalities")
accessibility.createOrReplaceTempView("accessibility_scores")
poi.createOrReplaceTempView("poi_counts")
property_values.createOrReplaceTempView("property_values")

In [6]:
spark.conf.set("spark.sql.shuffle.partitions", "4")

In [7]:
spark.sql("""
WITH ranked AS (
  SELECT
    municipality_id,
    municipality_name,
    canton,
    population,
    ROW_NUMBER() OVER (
      PARTITION BY canton
      ORDER BY population DESC, municipality_id ASC
    ) AS population_rank_in_canton
  FROM municipalities
)
SELECT *
FROM ranked
WHERE population_rank_in_canton <= 2
ORDER BY canton, population_rank_in_canton
""").explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Sort (10)
   +- Exchange (9)
      +- Filter (8)
         +- Window (7)
            +- WindowGroupLimit (6)
               +- Sort (5)
                  +- Exchange (4)
                     +- WindowGroupLimit (3)
                        +- Sort (2)
                           +- Scan csv  (1)


(1) Scan csv 
Output [4]: [municipality_id#17, municipality_name#18, canton#19, population#20]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/datasets/module_02/municipalities.csv]
ReadSchema: struct<municipality_id:int,municipality_name:string,canton:string,population:int>

(2) Sort
Input [4]: [municipality_id#17, municipality_name#18, canton#19, population#20]
Arguments: [canton#19 ASC NULLS FIRST, population#20 DESC NULLS LAST, municipality_id#17 ASC NULLS FIRST], false, 0

(3) WindowGroupLimit
Input [4]: [municipality_id#17, municipality_name#18, canton#19, population#20]
Arguments: [canton#19], [population#20 DESC NULLS LAST,

In [ ]:
query = """
SELECT m.canton,
       COUNT(*) AS municipalities,
       ROUND(AVG(a.accessibility_score), 1) AS avg_accessibility,
       ROUND(AVG(v.property_value_index), 1) AS avg_property_value
FROM municipalities m
JOIN accessibility_scores a USING (municipality_id)
JOIN property_values v USING (municipality_id)
WHERE m.population >= 10000
GROUP BY m.canton
ORDER BY avg_accessibility DESC
"""
spark.sql(query).show(truncate=False)

In [ ]:
spark.sql("EXPLAIN FORMATTED " + query).show(200, truncate=False)

Plan reading checklist: find file scans, pushed filters, join operators, aggregate operators, Exchange nodes, Sort nodes, and adaptive query execution markers.